# 猪特征工程 - 特征提取

标签：无猪 -0，活猪 -1，死猪 -2

原始特征：
- max_channel: 雷达检测到最大通道
- ideal_channel: 理想通道
- br_amp: 呼吸振幅（1 小时内提取的列表）
- br_cv: 呼吸的波动率（1 小时内提取的列表）
- up_down: 是否起卧（1 小时内提取的列表）

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 加载原始数据
df = pd.read_pickle('pig_features_data_all.pkl')
print(f"原始数据形状：{df.shape}")
print(f"标签分布：无猪 -0: {(df['label']==0).sum()}, 活猪 -1: {(df['label']==1).sum()}, 死猪 -2: {(df['label']==2).sum()}")

原始数据形状：(87749, 8)
标签分布：无猪 -0: 8644, 活猪 -1: 78951, 死猪 -2: 154


In [2]:
def extract_features(row):
    """从单个样本的时间序列中提取特征"""
    features = {}
    
    # 获取各个特征数组
    max_ch = row['max_channel']
    ideal_ch = row['ideal_channel']
    br_amp = row['br_amp']
    br_cv = row['br_cv']
    up_down = row['up_down']
    
    # ============ 1. 呼吸振幅 (br_amp) 特征 ============
    features['br_amp_mean'] = np.mean(br_amp)  # 平均呼吸振幅
    features['breathing_stop_ratio'] = np.mean(br_amp < 0.2)  # 呼吸停止检测
    features['breathing_high_ratio'] = np.mean(br_amp > 1.0)  # 高呼吸检测
    
    # ============ 2. 呼吸波动率 (br_cv) 特征 ============
    features['br_cv_mean'] = np.mean(br_cv)
    features['br_cv_high_ratio'] = np.mean(br_cv > 1.0)  # 高波动率占比
    features['br_cv_mid_ratio'] = np.mean((0.5 < br_cv) & (br_cv < 1))
    features['br_cv_low_ratio'] = np.mean(br_cv < 0.5)   # 低波动率占比
    
    # ============ 3. 通道相关特征 ============
    channel_match = (max_ch == ideal_ch).astype(float)
    features['channel_match_ratio'] = np.mean(channel_match)  # 通道匹配率
    max_ch_diff = np.diff(max_ch)
    features['max_channel_changes'] = np.sum(max_ch_diff != 0)  # 通道变化次数
    features['max_channel_no'] = np.mean((max_ch >= 21) & (max_ch <= 23)) #无猪通道
    features['max_channel_dead'] = np.mean((max_ch >= 12) &(max_ch <= 14)) #死猪通道
    features['max_channel_13'] = np.mean(max_ch == 13) #无猪/有猪通道
    features['ideal_channel_13'] = np.mean(ideal_ch == 13) 
    
    # ============ 4. 起卧行为特征 ============
    features['up_down_ratio'] = np.mean(up_down)  # 起卧时间占比
    up_down_diff = np.abs(np.diff(up_down))
    features['up_down_transitions'] = np.sum(up_down_diff > 0.5)  # 状态转换次数
    
    # 连续起卧时长
    consecutive_lengths = []
    current_length = 0
    for val in up_down:
        if val > 0.5:
            current_length += 1
        else:
            if current_length > 0:
                consecutive_lengths.append(current_length)
            current_length = 0
    if current_length > 0:
        consecutive_lengths.append(current_length)
    features['up_down_avg_duration'] = np.mean(consecutive_lengths) if consecutive_lengths else 0
    
    # ============ 5. 组合特征 ============
    features['activity_score'] = features['up_down_ratio'] * features['max_channel_changes']
    features['breathing_stability'] = features['br_amp_mean'] * (1 - features['br_cv_high_ratio'])
    
    # ============ 6. 基于规则分类的比例特征 ============
    # 调用calc_rule_rates函数获取规则分类比例
    rule_rates = calc_rule_rates(row)
    # 将规则分类比例添加到特征字典中
    features['rate_alive'] = rule_rates['rate_alive']
    features['rate_dead'] = rule_rates['rate_dead']
    features['rate_empty'] = rule_rates['rate_empty']
    
    return features

def calc_rule_rates(row):
    """基于规则的分类比例计算"""
    mc_arr = np.array(row['max_channel'])
    amp_arr = np.array(row['br_amp'])
    up_arr = np.array(row['up_down'])
    
    n = len(mc_arr)
    if n == 0:
        return {'rate_alive': 0, 'rate_dead': 0, 'rate_empty': 0}

    # 条件掩码
    # 活猪：(mc <= 20 and amp >= 0.5) or is_up
    mask_alive = ((mc_arr <= 20) & (amp_arr >= 0.5)) | (up_arr == 1)
    
    # 死猪：13 <= mc <= 15 and amp < 0.5
    mask_dead = (mc_arr >= 13) & (mc_arr <= 15) & (amp_arr < 0.5)
    
    # 无猪：其余情况
    mask_empty = ~(mask_alive | mask_dead)

    return {
        'rate_alive': np.sum(mask_alive) / n,
        'rate_dead': np.sum(mask_dead) / n,
        'rate_empty': np.sum(mask_empty) / n
    }

In [3]:
# 提取所有样本的特征
print("正在提取特征...")
all_features = []
for idx, row in df.iterrows():
    features = extract_features(row)
    features['label'] = row['label']
    features['file_path'] = row['file_path']  # 保留 file_path
    all_features.append(features)
    if (idx + 1) % 100 == 0:
        print(f"已处理 {idx + 1}/{len(df)} 样本")

features_df = pd.DataFrame(all_features)

正在提取特征...
已处理 100/87749 样本
已处理 200/87749 样本
已处理 300/87749 样本
已处理 400/87749 样本
已处理 500/87749 样本
已处理 600/87749 样本
已处理 700/87749 样本
已处理 800/87749 样本
已处理 900/87749 样本
已处理 1000/87749 样本
已处理 1100/87749 样本
已处理 1200/87749 样本
已处理 1300/87749 样本
已处理 1400/87749 样本
已处理 1500/87749 样本
已处理 1600/87749 样本
已处理 1700/87749 样本
已处理 1800/87749 样本
已处理 1900/87749 样本
已处理 2000/87749 样本
已处理 2100/87749 样本
已处理 2200/87749 样本
已处理 2300/87749 样本
已处理 2400/87749 样本
已处理 2500/87749 样本
已处理 2600/87749 样本
已处理 2700/87749 样本
已处理 2800/87749 样本
已处理 2900/87749 样本
已处理 3000/87749 样本
已处理 3100/87749 样本
已处理 3200/87749 样本
已处理 3300/87749 样本
已处理 3400/87749 样本
已处理 3500/87749 样本
已处理 3600/87749 样本
已处理 3700/87749 样本
已处理 3800/87749 样本
已处理 3900/87749 样本
已处理 4000/87749 样本
已处理 4100/87749 样本
已处理 4200/87749 样本
已处理 4300/87749 样本
已处理 4400/87749 样本
已处理 4500/87749 样本
已处理 4600/87749 样本
已处理 4700/87749 样本
已处理 4800/87749 样本
已处理 4900/87749 样本
已处理 5000/87749 样本
已处理 5100/87749 样本
已处理 5200/87749 样本
已处理 5300/87749 样本
已处理 5400/87749 样本
已处理 5500/87749 样本
已处理 5600/

In [4]:
# 特征与标签的相关性分析
from scipy.stats import pearsonr

print("\n=== 特征与标签的相关性（Top 20）===")
feature_cols = [c for c in features_df.columns if c != 'label' and c != 'file_path']
correlations = []
for col in feature_cols:
    corr, _ = pearsonr(features_df[col], features_df['label'])
    correlations.append((col, corr))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"{'特征':<30} {'相关系数':>10}")
print('-' * 45)
for col, corr in correlations[:20]:
    print(f"{col:<30} {corr:>10.4f}")


=== 特征与标签的相关性（Top 20）===
特征                                   相关系数
---------------------------------------------
max_channel_no                    -0.8646
rate_empty                        -0.8488
rate_alive                         0.8325
breathing_stop_ratio              -0.7805
breathing_high_ratio               0.7030
br_cv_low_ratio                   -0.5918
br_cv_mid_ratio                    0.5355
br_cv_mean                         0.4934
br_amp_mean                        0.4563
breathing_stability                0.4004
max_channel_dead                   0.3771
max_channel_changes                0.3715
br_cv_high_ratio                   0.2696
ideal_channel_13                   0.2072
max_channel_13                     0.2026
up_down_transitions                0.1708
up_down_ratio                      0.1616
activity_score                     0.1424
up_down_avg_duration               0.1269
channel_match_ratio               -0.1194


In [5]:
# ==================== XGBoost 模型训练 ====================
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# 准备数据
feature_cols = [c for c in features_df.columns if c not in ['label', 'file_path']]
X = features_df[feature_cols].values
y = features_df['label'].values
file_paths = features_df['file_path'].values  # 保存 file_path

# 切分训练集和测试集（分层抽样，保持类别比例）
# 同时分割 X, y, 和 file_paths
X_train, X_test, y_train, y_test, file_paths_train, file_paths_test = train_test_split(
    X, y, file_paths,
    test_size=0.2,  # 20% 作为测试集
    random_state=42,
    stratify=y  # 分层抽样
)

print(f"训练集样本数：{len(X_train)}, 测试集样本数：{len(X_test)}")
print(f"训练集标签分布：0={sum(y_train==0)}, 1={sum(y_train==1)}, 2={sum(y_train==2)}")
print(f"测试集标签分布：0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")

训练集样本数：70199, 测试集样本数：17550
训练集标签分布：0=6915, 1=63161, 2=123
测试集标签分布：0=1729, 1=15790, 2=31


In [6]:
import pickle
with open('test_file.pkl', 'wb') as f:  # 'wb'表示二进制写入
    pickle.dump(list(file_paths_test), f)

In [42]:
# 训练 XGBoost 模型
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    subsample = 0.6,
    colsample_bytree = 0.9
)

model.fit(X_train, y_train)

# 预测
y_pred = model.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f"\n测试集准确率：{accuracy:.4f}")

# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("\n=== 混淆矩阵 (3×3) ===")
print("预测 →")
print("真实 ↓    无猪(0)  活猪(1)  死猪(2)")
print(f"无猪(0)    {cm[0][0]:^5}   {cm[0][1]:^5}   {cm[0][2]:^5}")
print(f"活猪(1)    {cm[1][0]:^5}   {cm[1][1]:^5}   {cm[1][2]:^5}")
print(f"死猪(2)    {cm[2][0]:^5}   {cm[2][1]:^5}   {cm[2][2]:^5}")

# 分类报告
print("\n=== 分类报告 ===")
print(classification_report(y_test, y_pred, target_names=['无猪(0)', '活猪(1)', '死猪(2)']))


测试集准确率：0.9947

=== 混淆矩阵 (3×3) ===
预测 →
真实 ↓    无猪(0)  活猪(1)  死猪(2)
无猪(0)    1659     70       0  
活猪(1)      7     15783     0  
死猪(2)      3      13      15  

=== 分类报告 ===
              precision    recall  f1-score   support

       无猪(0)       0.99      0.96      0.98      1729
       活猪(1)       0.99      1.00      1.00     15790
       死猪(2)       1.00      0.48      0.65        31

    accuracy                           0.99     17550
   macro avg       1.00      0.81      0.88     17550
weighted avg       0.99      0.99      0.99     17550



In [43]:
# 获取混淆矩阵每个格子对应的样本索引和文件路径
# 格式：samples[真实标签][预测标签] = 文件路径列表

from collections import defaultdict

# 创建字典存储每个格子对应的样本
samples = defaultdict(lambda: defaultdict(list))

for i, (true_label, pred_label, file_path) in enumerate(zip(y_test, y_pred, file_paths_test)):
    samples[true_label][pred_label].append(file_path)

# 打印每个格子对应的样本数量和文件路径
label_names = {0: '无猪', 1: '活猪', 2: '死猪'}

print("=== 混淆矩阵样本列表 ===\n")
for true_label in sorted(samples.keys()):
    for pred_label in sorted(samples[true_label].keys()):
        file_list = samples[true_label][pred_label]
        print(f"真实:{label_names[true_label]}({true_label}) → 预测:{label_names[pred_label]}({pred_label})")
        print(f"  数量: {len(file_list)}")
        print(f"  文件路径: {file_list}\n")

# 保存到文件
# import pickle
# with open('confusion_matrix_samples.pkl', 'wb') as f:
#     pickle.dump(dict(samples), f)
# print("样本列表已保存到 confusion_matrix_samples.pkl")

=== 混淆矩阵样本列表 ===

真实:无猪(0) → 预测:无猪(0)
  数量: 1659
  文件路径: ['/data1/chenming40/alive_detect/no_pig/13113-2025050605.hdf5', '/data1/chenming40/alive_detect/no_pig/30773-2025090814.hdf5', '/data1/chenming40/alive_detect/no_pig/30306-2025052000.hdf5', '/data1/chenming40/alive_detect/no_pig/13105-2025050908.hdf5', '/data1/chenming40/alive_detect/no_pig/13106-2025052002.hdf5', '/data1/chenming40/alive_detect/no_pig/30306-2025051905.hdf5', '/data1/chenming40/alive_detect/no_pig/29773-2025051910.hdf5', '/data1/chenming40/alive_detect/no_pig/29554-2025052002.hdf5', '/data1/chenming40/alive_detect/no_pig/29773-2025052002.hdf5', '/data1/chenming40/alive_detect/no_pig/30067-2025051705.hdf5', '/data1/chenming40/alive_detect/no_pig/13113-2025051710.hdf5', '/data1/chenming40/alive_detect/no_pig/13104-2025051802.hdf5', '/data1/chenming40/alive_detect/no_pig/13105-2025050404.hdf5', '/data1/chenming40/alive_detect/no_pig/29776-2025051805.hdf5', '/data1/chenming40/alive_detect/no_pig/29799-2025051800.hdf5

In [ ]:
# 保存特征
features_df.to_pickle('engineered_features.pkl')
print("\n特征已保存到 engineered_features.pkl")
print(f"特征文件形状：{features_df.shape}")